# Effective-action term-by-term comparison

This notebook checks the Regge-sector contribution and the two non-Regge quadratic pieces used in the manuscript: the $(\log E_h)^2$ part and the transverse non-Regge part.

The comparison here follows the manuscript structure:

1. build the solver and effective-stage data,
2. use `compute_boundary_taylor_terms(data, gamma_value)` to obtain the Gaussian source blocks $B_I$, $C_I$, $\mathfrak h_{IJ}$, and $\mathcal I_{\rm bdy}^{(2)}$,
3. compare the full Regge-sector left-hand side with the Regge expansion,
4. compare the direct $(\log E_h)^2$ and transverse Gaussian blocks with their kernel forms.

$$
\left[\mathcal I_{\rm eff}(\ell)-\mathring{S}'\right]^{(2)}
=iS_{\rm Regge}^{(2)}
-\frac{\gamma^2}{8}\,\varepsilon^\top K\varepsilon,
\qquad
K=\rho-\rho A^{-1}\kappa A^{-1}\rho.
$$

In [1]:
using Pkg
root_path = joinpath(pwd(), "..")
Pkg.activate(root_path)
Pkg.instantiate()

using EffectiveSpinfoam
using LorentzianSimplexSolver
using LinearAlgebra
using SymEngine

  Activating project at `~/Documents/Work/effective-spinfoam/code/Effective-Spinfoam`


## 1. Geometry and parameters

The example below is the double-$\Delta_3$ complex used in the numerical tests.  The precision and Immirzi parameter can be changed here without changing the rest of the notebook.

In [2]:
simplices = [[1,2,3,4,6], [1,2,3,5,6], [1,2,4,5,6],[1,2,3,4,7], [1,2,3,5,7], [1,2,4,5,7]]

coords_lines = [
    "0, 0, 0, 0",
    "-0.068000000000000005, -0.21988127663727278, -0.5316227766016838, -1.3316227766016839",
    "0, 0, 0, -3.398088489694245",
    "-0.24028114141347542, -0.6936319083813028, -0.9809436521275706, -1.6990442448471226",
    "0, 0, -2.942830956382712, -1.6990442448471226",
    "0, -2.7745276335252114, -0.9809436521275706, -1.6990442448471226",
    "-2.4696884592430974, -3.893218630529324, -1.3565336794679874, -1.9090667752920147",
]

ScalarT = Float64
# ScalarT = BigFloat
tol = LorentzianSimplexSolver.configure_precision!(ScalarT; precision=80)

gamma_value = ScalarT(1.0);
vertex_coords = build_vertex_coordinates(simplices, coords_lines, ScalarT);

## 2. Workflow data

`build_solver_data` calls `LorentzianSimplexSolver`.  `build_effective_data` adds the effective-stage information.  The important simplification in this notebook is that the boundary Taylor expansion is now computed by one call to `compute_boundary_taylor_terms`.

In [3]:
solver = build_solver_data(simplices, vertex_coords; bulk_sum_form=false);
data = build_effective_data(solver, simplices, vertex_coords);

blocks = evaluate_effective_blocks(data, gamma_value);
boundary_terms = compute_boundary_taylor_terms(data, gamma_value; blocks=blocks, tol=tol);
matrices = compute_effective_matrices(data, gamma_value; blocks=blocks);

The objects used below are:

- `boundary_terms.B` and `boundary_terms.C`: the Gaussian source blocks in the $X$ directions,
- `boundary_terms.h_frak`: the Regge-sector $X$-Hessian $\mathfrak h$,
- `matrices.dlogEh_dX`: the code version of $V_{I,h}=[\partial_{X_I}\log E_h]_0$ with indices stored as `(h, I)`,
- `matrices.transverse_basis`: the transverse basis $\hat e^i_h$,
- `matrices.rho` and `matrices.kappa`: the Schur-complement blocks $\rho$ and $\kappa$,
- `matrices.K_kernel`: the kernel $K_{hh'}$.


In [4]:
nh = length(data.eta_h_vertices)
nl = length(data.perturb_edges)
nX = size(boundary_terms.h_frak, 1)
nt = size(matrices.transverse_basis, 2)

println("number of bulk hinges nh     = ", nh)
println("number of length variables nl = ", nl)
println("number of X variables nX      = ", nX)
println("number of transverse modes nt = ", nt)

number of bulk hinges nh     = 5
number of length variables nl = 2
number of X variables nX      = 237
number of transverse modes nt = 4


## 3. Gaussian blocks and common matrices

The Regge-sector Gaussian integrates only the $X$ variables. After the $(\log E_h)^2$ term and the transverse variables $t_i$ are included, the manuscript ordering is $\mathcal X=(t,X)$:

$$
\mathcal H=
\begin{pmatrix}
0 & D\\
D^\top & H
\end{pmatrix},
\qquad
H=\mathfrak h+VAV^\top,
\qquad
D=\hat e^\top V^\top,
\qquad
W=-V^\top H^{-1}V,
\qquad
G=-DH^{-1}D^\top=\hat e^\top W\hat e.
$$

In the code, `Vt` stores $V^\top$ as an array with indices $(h,I)$, and `e_hat` stores $\hat e$ with indices $(h,i)$. The code evaluates $G=-DH^{-1}D^\top$ directly and does not store $W$ separately. The array `calH` uses the computational ordering $(X,t)$, so its $X$-$X$ block occupies the first `nX` rows and columns.

In [5]:
source_X = boundary_terms.B + boundary_terms.C
h_frak = boundary_terms.h_frak
inv_h_frak = boundary_terms.inv_h_frak
Vt = matrices.dlogEh_dX
e_hat = matrices.transverse_basis

A = matrices.A
invA = inv(A)

H = matrices.H
invH = inv(H)
D = matrices.D
G = matrices.G

calH = matrices.calH
inv_calH = inv(calH)
calH_inv_XX = inv_calH[1:nX, 1:nX]

result = compute_effective_terms(data, gamma_value);

In [6]:
function print_term_summary(name, value; show_matrix=false)
    println(name)
    show(stdout, "text/plain", value)
    println()
    return value
end

max_abs_entry(value::Number) = abs(value)
max_abs_entry(value) = maximum(abs.(value))

function report_comparison(name, lhs, rhs)
    diff = lhs - rhs
    println(name)

    println("lhs:")
    show(stdout, "text/plain", lhs)
    println()

    println("rhs:")
    show(stdout, "text/plain", rhs)
    println()

    println("max |lhs - rhs| = ", max_abs_entry(diff))

    return diff
end;

## 4. Regge part

The first line of the Gaussian decomposition contains the Regge-sector contribution.  We compare the full left-hand side with the Regge expansion,

$$
-\frac12\mathcal B_X^\top \mathfrak h^{-1}\mathcal B_X
+
\mathcal I_{\rm bdy}^{(2)}
=
iS_{\rm Regge}^{(2)}.
$$

The superscript $(2)$ denotes the expansion through second order and therefore includes the boundary term linear in $\delta\ell$. In the code, the degree-one coefficient vector and the degree-two coefficient matrix are flattened and concatenated only for this comparison.

In [7]:
regge_lhs_linear = vec(transpose(boundary_terms.I_bdy_linear))
regge_lhs_quadratic =
    -1 / 2 * transpose(source_X) * inv_h_frak * source_X +
    boundary_terms.I_bdy_quadratic

regge_lhs = vcat(regge_lhs_linear, vec(regge_lhs_quadratic))
regge_rhs = vcat(vec(result.linear_regge), vec(result.quadratic_regge))

print_term_summary("Regge left-hand side coefficients", regge_lhs)
println()
print_term_summary("Regge expansion coefficients", regge_rhs)
println()

regge_difference = report_comparison(
    "Regge part: left-hand side = Regge expansion",
    regge_lhs,
    regge_rhs,
);

Regge left-hand side coefficients
6-element Vector{ComplexF64}:
  1.9317807790331185e-11 + 1.7455977726550014e-12im
  -8.902582206068198e-12 - 2.4781224413740857im
  1.4504842603512925e-11 - 134.08908832298485im
  -6.579999923353035e-12 + 152.53900780656306im
 -6.5221390819962744e-12 + 152.53900780656303im
   1.956435013994451e-12 - 117.7528674387721im

Regge expansion coefficients
6-element Vector{Any}:
  0.0 + 0.0im
  0.0 - 2.478122441362764im
 -0.0 - 134.08908832297993im
  0.0 + 152.53900780656636im
  0.0 + 152.53900780656585im
 -0.0 - 117.75286743877362im

Regge part: left-hand side = Regge expansion
lhs:
6-element Vector{ComplexF64}:
  1.9317807790331185e-11 + 1.7455977726550014e-12im
  -8.902582206068198e-12 - 2.4781224413740857im
  1.4504842603512925e-11 - 134.08908832298485im
  -6.579999923353035e-12 + 152.53900780656306im
 -6.5221390819962744e-12 + 152.53900780656303im
   1.956435013994451e-12 - 117.7528674387721im
rhs:
6-element Vector{Any}:
  0.0 + 0.0im
  0.0 - 2.4781224413

## 5. $(\log E_h)^2$ part

The second Gaussian block is

$$
\frac12\mathcal B_X^\top \mathfrak h^{-1}V\rho V^\top\mathfrak h^{-1}\mathcal B_X.
$$

Using

$$
V^\top\mathfrak h^{-1}\mathcal B_X
=
-\frac{i\gamma}{2}\varepsilon,
\qquad
\varepsilon=(D_\ell\varepsilon)\delta\ell,
$$

the same block should equal

$$
-\frac{\gamma^2}{8}
\varepsilon^\top\rho\varepsilon.
$$
The code compares the corresponding length-space coefficient matrices before contraction with $\delta\ell$.

In [8]:
log_square_gaussian =
    1 / 2 * transpose(source_X) * inv_h_frak * transpose(Vt) *
    matrices.rho * Vt * inv_h_frak * source_X

log_square_kernel =
    -gamma_value^2 / 8 * transpose(data.DεDl) * matrices.rho * data.DεDl

print_term_summary("Direct (log E_h)^2 Gaussian block", log_square_gaussian)
println()
print_term_summary("Kernel (log E_h)^2 block", log_square_kernel)
println()

log_square_difference = report_comparison(
    "(log E_h)^2 part: direct block = kernel block",
    log_square_gaussian,
    log_square_kernel,
);

Direct (log E_h)^2 Gaussian block
2×2 Matrix{ComplexF64}:
 -13.0835+133.95im   8.32886-146.15im
  8.32886-146.15im  -26.4625+128.336im

Kernel (log E_h)^2 block
2×2 Matrix{ComplexF64}:
 -13.0835+133.95im   8.32886-146.15im
  8.32886-146.15im  -26.4625+128.336im

(log E_h)^2 part: direct block = kernel block
lhs:
2×2 Matrix{ComplexF64}:
 -13.0835+133.95im   8.32886-146.15im
  8.32886-146.15im  -26.4625+128.336im
rhs:
2×2 Matrix{ComplexF64}:
 -13.0835+133.95im   8.32886-146.15im
  8.32886-146.15im  -26.4625+128.336im
max |lhs - rhs| = 2.253017357041479e-11


## 6. Transverse part

The third Gaussian block comes from integrating the transverse variables $t_i$:

$$
-\frac12\mathcal B_X^\top
H^{-1}D^\top G^{-1}DH^{-1}
\mathcal B_X.
$$

With

$$
G=-DH^{-1}D^\top,
\qquad
\kappa=\hat e G^{-1}\hat e^\top,
$$

this becomes

$$
\frac{\gamma^2}{8}
\varepsilon^\top
\rho A^{-1}\kappa A^{-1}\rho
\varepsilon.
$$
The numerical comparison again uses the coefficient matrix in length space.

In [9]:
transverse_gaussian =
    -1 / 2 * transpose(source_X) * invH * transpose(D) *
    inv(G) * D * invH * source_X

transverse_kernel =
    gamma_value^2 / 8 * transpose(data.DεDl) * matrices.rho * invA *
    matrices.kappa * invA * matrices.rho * data.DεDl

print_term_summary("Direct transverse Gaussian block", transverse_gaussian)
println()
print_term_summary("Kernel transverse block", transverse_kernel)
println()

transverse_difference = report_comparison(
    "Transverse part: direct block = kernel block",
    transverse_gaussian,
    transverse_kernel,
);

Direct transverse Gaussian block
2×2 Matrix{ComplexF64}:
   9.5687+0.225984im  -6.03918-2.15435im
 -6.03918-2.15435im     1.1686-1.57255im

Kernel transverse block
2×2 Matrix{ComplexF64}:
   9.5687+0.225984im  -6.03918-2.15435im
 -6.03918-2.15435im     1.1686-1.57255im

Transverse part: direct block = kernel block
lhs:
2×2 Matrix{ComplexF64}:
   9.5687+0.225984im  -6.03918-2.15435im
 -6.03918-2.15435im     1.1686-1.57255im
rhs:
2×2 Matrix{ComplexF64}:
   9.5687+0.225984im  -6.03918-2.15435im
 -6.03918-2.15435im     1.1686-1.57255im
max |lhs - rhs| = 1.4359986789242949e-11


## 7. Final coefficient output

The API stores the degree-one and degree-two pieces of $iS_{\rm Regge}^{(2)}$ separately in `linear_regge` and `quadratic_regge`. The field `non_regge_correction` stores the length-space matrix $-\gamma^2(D_\ell\varepsilon)^\top K(D_\ell\varepsilon)/8$.

In [10]:
print_term_summary("Linear Regge part", result.linear_regge)
println()
print_term_summary("Quadratic Regge term", result.quadratic_regge)
println()
print_term_summary("Non-Regge correction term", result.non_regge_correction)


Linear Regge part
2-element Vector{Any}:
 0.0 + 0.0im
 0.0 - 2.478122441362764im

Quadratic Regge term
2×2 Matrix{ComplexF64}:
 -0.0-134.089im   0.0+152.539im
  0.0+152.539im  -0.0-117.753im

Non-Regge correction term
2×2 Matrix{ComplexF64}:
 -3.51483+134.176im   2.28968-148.304im
  2.28968-148.304im  -25.2939+126.764im


2×2 Matrix{ComplexF64}:
 -3.51483+134.176im   2.28968-148.304im
  2.28968-148.304im  -25.2939+126.764im

In [11]:
delta_lengths = fill(ScalarT(1e-4),length(data.perturb_edges),)

linear_difference =
    sum(result.linear_regge .* delta_lengths)

quadratic_regge_difference =
    transpose(delta_lengths) *
    result.quadratic_regge *
    delta_lengths

non_regge_difference =
    transpose(delta_lengths) *
    result.non_regge_correction *
    delta_lengths

quadratic_difference =
    quadratic_regge_difference + non_regge_difference

analytic_action_difference =
    linear_difference + quadratic_difference

println("Length perturbations:")
for (edge, delta_l) in zip(data.perturb_edges, delta_lengths)
    println("  δl_$edge = $delta_l")
end

println()
println("Linear action difference     = ", linear_difference)
println("Quadratic Regge difference  = ", quadratic_regge_difference)
println("Non-Regge difference        = ", non_regge_difference)
println("Total quadratic difference  = ", quadratic_difference)
println("Analytic action difference  = ", analytic_action_difference)

Length perturbations:
  δl_[1, 2] = 0.0001
  δl_[1, 3] = 0.0001

Linear action difference     = 0.0 - 0.00024781224413627644im
Quadratic Regge difference  = 0.0 + 5.323605985137867e-7im
Non-Regge difference        = -2.4229414854383194e-7 - 3.566818096351375e-7im
Total quadratic difference  = -2.4229414854383194e-7 + 1.7567878887864919e-7im
Analytic action difference  = -2.4229414854383194e-7 - 0.0002476365653473978im
